In [ ]:
# Xlmr-Bert Fine-Tuning (Cross Validation)

# Part 1: Install Required Libraries

In [ ]:
!pip uninstall -y transformers torchvision torch torchaudio accelerate -q

!pip install -q \
torch==2.7.1 \
torchvision==0.22.1 \
torchaudio==2.7.1 \
--index-url https://download.pytorch.org/whl/cu128

!pip install -q \
transformers==4.53.3 \
accelerate==1.8.1 \
datasets==4.0.0 \
evaluate==0.4.5 \
sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 138.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 235.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 187.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 34.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 60.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 73.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 56.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 77.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 MB 100.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 71.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 MB 68.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 66.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Mount Google Drive to access the pretrained checkpoint, datasets, and save fine-tuned models
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Imports Libraries
import os
import random
import warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

warnings.filterwarnings("ignore")

In [ ]:
# Detect and select the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device : cuda
Tesla T4


In [ ]:
# Set random seed for reproducibility across random, numpy, and torch
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Enable cuDNN autotuning and TF32 matmul/conv for faster GPU performance
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# Path to the extracted MLM checkpoint, and where fine-tuned fold models will be saved
MODEL_PATH = "/content/drive/MyDrive/XLM-R Cross Validation/Best_XLMR_MLM8"

SAVE_DIR = "/content/drive/MyDrive/XLM-R Cross Validation/XLMR_CV"

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Load the tokenizer from the MLM checkpoint and set the max sequence length
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

MAX_LEN = 64

In [ ]:
# Tokenization function: truncates to MAX_LEN, padding deferred to the data collator
def tokenize(batch):

    return tokenizer(

        batch["sentence"],

        truncation=True,

        max_length=MAX_LEN,

        padding=False

    )

In [ ]:
# Quick sanity check: tokenize a sample sentence
print(tokenizer("Climate change is real."))

{'input_ids': [0, 135987, 67, 15549, 83, 2773, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


## Part 2: Load and Prepare the Dataset

In [ ]:
# Load the raw per-language CSV files
english_df = pd.read_csv("/content/English_train_data.csv")

hindi_df = pd.read_csv("/content/english_to_hindi.csv")

bengali_df = pd.read_csv("/content/english_to_bengali.csv")

none_df = pd.read_csv("/content/non_climate_none.csv")

In [ ]:
# Keep only the text and label columns, and rename them to a common
# "sentence" / "label" schema so all datasets can be concatenated
hindi = hindi_df[["hindi","label"]].copy()
hindi.columns = ["sentence","label"]

bengali = bengali_df[["bengali","label"]].copy()
bengali.columns = ["sentence","label"]

english = english_df[["sentence","label"]].copy()

none_df = none_df[["sentence","label"]].copy()

In [ ]:
# Concatenate all language datasets into a single training dataframe
train_df = pd.concat(

    [
        english,
        hindi,
        bengali,
        none_df
    ],

    ignore_index=True

)

In [ ]:
# Remove rows with missing values
train_df = train_df.dropna()
train_df = train_df.reset_index(drop=True)

In [ ]:
# Remove duplicate sentences to avoid data leakage/repetition
train_df = train_df.drop_duplicates(subset="sentence")

train_df = train_df.reset_index(drop=True)

In [ ]:
# Shuffle the merged dataset so languages/classes are interleaved
train_df = train_df.sample(

    frac=1,

    random_state=42

).reset_index(drop=True)

In [ ]:
# Preview the merged dataset
print(train_df.head())

                                            sentence    label
0  बहरहाल, ग्लोबल वार्मिंग उन्माद के पीछे एक वास्...  AGAINST
1  : চীন খুব সদয়ভাবে ট্রাম্পকে ব্যাখ্যা করেছে যে...    FAVOR
2  उस पर रखें। इस सप्ताह पहले ऐसा ही कुछ सुना था....     NONE
3  আপনি কীভাবে আমাকে বলবেন যে জলবায়ু পরিবর্তন/গ্...     NONE
4  Maybe it's like climate change making all the ...     NONE


In [ ]:
# Manually map string labels to integer ids
label2id = {

    "AGAINST":0,

    "FAVOR":1,

    "NONE":2

}

id2label = {

    0:"AGAINST",

    1:"FAVOR",

    2:"NONE"

}

train_df["label"] = train_df["label"].map(label2id)

In [ ]:
# Verify the label encoding was applied correctly
print(train_df["label"].value_counts())

print()

print(train_df.head())

label
2    22650
0    12610
1    12605
Name: count, dtype: int64

                                            sentence  label
0  बहरहाल, ग्लोबल वार्मिंग उन्माद के पीछे एक वास्...      0
1  : চীন খুব সদয়ভাবে ট্রাম্পকে ব্যাখ্যা করেছে যে...      1
2  उस पर रखें। इस सप्ताह पहले ऐसा ही कुछ सुना था....      2
3  আপনি কীভাবে আমাকে বলবেন যে জলবায়ু পরিবর্তন/গ্...      2
4  Maybe it's like climate change making all the ...      2


In [ ]:
# Check class balance as proportions of the total dataset
print(train_df["label"].value_counts(normalize=True))

label
2    0.473206
0    0.263449
1    0.263345
Name: proportion, dtype: float64


In [ ]:
# Analyze tokenized sentence lengths (in subword tokens) to inform truncation settings
lengths = train_df["sentence"].apply(

    lambda x: len(tokenizer.tokenize(str(x)))

)

print(lengths.describe())

Token indices sequence length is longer than the specified maximum sequence length for this model (866 > 512). Running this sequence through the model will result in indexing errors


count    47865.000000
mean        31.292427
std         28.370140
min          1.000000
25%         20.000000
50%         28.000000
75%         37.000000
max       1451.000000
Name: sentence, dtype: float64


In [ ]:
# Check the maximum token length and 95th percentile to sanity-check MAX_LEN
print("Maximum Tokens :", lengths.max())

print("95 Percentile :", np.percentile(lengths,95))

Maximum Tokens : 1451
95 Percentile : 56.0


In [ ]:
# Print a snapshot of dataset size and class distribution before text cleaning
print("="*60)

print("Total Samples :", len(train_df))

print()

print(train_df["label"].value_counts())

print("="*60)

Total Samples : 47865

label
2    22650
0    12610
1    12605
Name: count, dtype: int64


In [ ]:
# Save the cleaned dataset to Drive so it can be reloaded later
train_df.to_csv(

    "/content/drive/MyDrive/clean_train.csv",

    index=False

)

PART 3 — 5-Fold Stratified Cross Validation

In [ ]:
# Define a 5-fold stratified split so each fold preserves the overall class distribution
NUM_FOLDS = 5

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

In [ ]:
# Initialize lists to collect per-fold metrics
fold_accuracy = []
fold_precision = []
fold_recall = []
fold_f1 = []

In [ ]:
# Reset the metric lists again right before running the full cross-validation loop
fold_accuracy = []
fold_precision = []
fold_recall = []
fold_f1 = []

print("Metric lists re-initialized.")

Metric lists re-initialized.


In [ ]:
# Reset MODEL_PATH to the base pre-trained checkpoint before the CV loop
MODEL_PATH = "/content/drive/MyDrive/XLM-R Cross Validation/Best_XLMR_MLM8"
print(f"MODEL_PATH reset to: {MODEL_PATH}")

MODEL_PATH reset to: /content/drive/MyDrive/XLM-R Cross Validation/Best_XLMR_MLM8


In [ ]:
# Start Cross Validation Loop
for fold, (train_idx, val_idx) in enumerate(
        skf.split(train_df["sentence"], train_df["label"])):

    print("="*70)
    print(f"Fold {fold+1}/{NUM_FOLDS}")
    print("="*70)

    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)

    print("Training Samples :", len(train_fold))
    print("Validation Samples :", len(val_fold))

Fold 1/5
Training Samples : 38292
Validation Samples : 9573
Fold 2/5
Training Samples : 38292
Validation Samples : 9573
Fold 3/5
Training Samples : 38292
Validation Samples : 9573
Fold 4/5
Training Samples : 38292
Validation Samples : 9573
Fold 5/5
Training Samples : 38292
Validation Samples : 9573


In [ ]:
# Compute class weights for this trial fold (the last fold from the loop above).
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_fold["label"]),
    y=train_fold["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

print(class_weights)

tensor([1.2653, 1.2658, 0.7044], device='cuda:0')


In [ ]:
# Convert the trial fold's train/val splits into Hugging Face Datasets
train_dataset = Dataset.from_pandas(
    train_fold,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_fold,
    preserve_index=False
)

In [ ]:
# Tokenize the trial fold's datasets
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/38292 [00:00<?, ? examples/s]

Map:   0%|          | 0/9573 [00:00<?, ? examples/s]

In [ ]:
# Drop the raw text column now that it has been tokenized
train_dataset = train_dataset.remove_columns(
    ["sentence"]
)

val_dataset = val_dataset.remove_columns(
    ["sentence"]
)

In [ ]:
# Rename "label" to "labels", the column name expected by the Trainer
train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

val_dataset = val_dataset.rename_column(
    "label",
    "labels"
)

In [ ]:
# Set dataset output format to PyTorch tensors
train_dataset.set_format("torch")

val_dataset.set_format("torch")

In [ ]:
# Inspect the resulting trial-fold datasets
print(train_dataset)

print(val_dataset)

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 38292
})
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 9573
})


PART 4 — Fine-Tuning with Cross Validation

In [ ]:
# Dynamic padding collator
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

In [ ]:
# Function passed to the Trainer to compute accuracy/precision/recall/macro-F1
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Weighted Trainer
from transformers import Trainer

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights
        )

        loss = loss_fn(
            logits,
            labels
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:
# Training Arguments
training_args = TrainingArguments(

    output_dir="./Results",

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    num_train_epochs=5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    gradient_accumulation_steps=2,

    learning_rate=2e-5,

    weight_decay=0.01,

    warmup_ratio=0.1,

    fp16=True,

    save_total_limit=1,

    report_to="none",

    seed=42
)

In [ ]:
# Early Stopping
early_stop = EarlyStoppingCallback(

    early_stopping_patience=2,

    early_stopping_threshold=0.001
)

In [ ]:
# Load a Fresh Model for Each Fold
config = AutoConfig.from_pretrained(MODEL_PATH)

config.num_labels = 3

config.label2id = label2id

config.id2label = id2label

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_PATH,

    config=config,

    ignore_mismatched_sizes=True

)

model.to(device)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/XLM-R Cross Validation/Best_XLMR_MLM8 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [ ]:
# Freeze Embedding Layer
for param in model.roberta.embeddings.parameters():
    param.requires_grad = False

In [ ]:
# Start Full Cross Validation Loop (including training, evaluation, and saving)
for fold, (train_idx, val_idx) in enumerate(
        skf.split(train_df["sentence"], train_df["label"])):

    print("="*70)
    print(f"Fold {fold+1}/{NUM_FOLDS}")
    print("="*70)

    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)

    print("Training Samples :", len(train_fold))
    print("Validation Samples :", len(val_fold))

    # Compute Class Weights inside every fold
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_fold["label"]),
        y=train_fold["label"]
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float
    ).to(device)

    print("Class Weights for Fold ", fold+1, ":", class_weights)

    # Convert to HuggingFace Dataset
    train_dataset = Dataset.from_pandas(
        train_fold,
        preserve_index=False
    )

    val_dataset = Dataset.from_pandas(
        val_fold,
        preserve_index=False
    )

    # Tokenize
    train_dataset = train_dataset.map(
        tokenize,
        batched=True
    )

    val_dataset = val_dataset.map(
        tokenize,
        batched=True
    )

    # Remove Sentence Column
    train_dataset = train_dataset.remove_columns(
        ["sentence"]
    )

    val_dataset = val_dataset.remove_columns(
        ["sentence"]
    )

    # Rename Label Column
    train_dataset = train_dataset.rename_column(
        "label",
        "labels"
    )

    val_dataset = val_dataset.rename_column(
        "label",
        "labels"
    )

    # Torch Format
    train_dataset.set_format("torch")
    val_dataset.set_format("torch")

    # Check Dataset (optional, for verification within the loop)
    print("Train Dataset for Fold ", fold+1, ":", train_dataset)
    print("Validation Dataset for Fold ", fold+1, ":", val_dataset)

    # Load a Fresh Model for Each Fold
    config = AutoConfig.from_pretrained(MODEL_PATH)
    config.num_labels = 3
    config.label2id = label2id
    config.id2label = id2label

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        config=config,
        ignore_mismatched_sizes=True
    )
    model.to(device)

    # Freeze Embeddings
    for param in model.roberta.embeddings.parameters():
        param.requires_grad = False

    # Create Trainer
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[early_stop]
    )

    # Train
    print(f"\nStarting training for Fold {fold+1}...")
    trainer.train()
    print(f"Finished training for Fold {fold+1}.")

    # Evaluate
    print(f"\nStarting evaluation for Fold {fold+1}...")
    metrics = trainer.evaluate()
    print(f"Evaluation Metrics for Fold {fold+1}: {metrics}")

    fold_accuracy.append(metrics["eval_accuracy"])
    fold_precision.append(metrics["eval_precision"])
    fold_recall.append(metrics["eval_recall"])
    fold_f1.append(metrics["eval_f1"])

    # Save Each Fold
    fold_dir = os.path.join(
        SAVE_DIR,
        f"Fold_{fold+1}"
    )

    trainer.save_model(fold_dir)
    tokenizer.save_pretrained(fold_dir)

    print(f"Model Saved -> {fold_dir}\n")

print("="*70)
print("Cross-validation loop completed.")
print("="*70)


Fold 1/5
Training Samples : 38292
Validation Samples : 9573
Class Weights for Fold  1 : tensor([1.2653, 1.2658, 0.7044], device='cuda:0')


Map:   0%|          | 0/38292 [00:00<?, ? examples/s]

Map:   0%|          | 0/9573 [00:00<?, ? examples/s]

Train Dataset for Fold  1 : Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 38292
})
Validation Dataset for Fold  1 : Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 9573
})


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/XLM-R Cross Validation/Best_XLMR_MLM8 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at /content/drive/MyDrive/XLM-R Cross Validation/Best_XLMR_MLM8 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting training for Fold 5...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.774800,0.601448,0.732059,0.731242,0.748840,0.728642
2,0.533100,0.563922,0.769665,0.761741,0.785933,0.765937
3,0.387000,0.553633,0.781573,0.775566,0.798838,0.778984
4,0.280900,0.573195,0.801421,0.790972,0.811410,0.797482
5,0.210600,0.625904,0.809151,0.799348,0.812050,0.804219


Finished training for Fold 5.

Starting evaluation for Fold 5...


Evaluation Metrics for Fold 5: {'eval_loss': 0.625903844833374, 'eval_accuracy': 0.8091507364462551, 'eval_precision': 0.7993482649226434, 'eval_recall': 0.8120498576450298, 'eval_f1': 0.8042189898108676, 'eval_runtime': 9.7255, 'eval_samples_per_second': 984.324, 'eval_steps_per_second': 30.847, 'epoch': 5.0}
Model Saved -> /content/drive/MyDrive/XLM-R Cross Validation/XLMR_CV/Fold_5

Cross-validation loop completed.


In [ ]:
# Cross Validation Summary

results_df = pd.DataFrame({
    "Fold": range(1, NUM_FOLDS + 1),
    "Accuracy": fold_accuracy,
    "Precision": fold_precision,
    "Recall": fold_recall,
    "Macro_F1": fold_f1
})

print(results_df)

   Fold  Accuracy  Precision    Recall  Macro_F1
0     1  0.802570   0.792612  0.812805  0.799547
1     2  0.812807   0.803408  0.822187  0.809420
2     3  0.817299   0.807965  0.823885  0.813613
3     4  0.818448   0.809869  0.822644  0.814358
4     5  0.809151   0.799348  0.812050  0.804219


In [ ]:
# Average Performance
print("="*70)
print("5-Fold Cross Validation Results")
print("="*70)

print(f"Average Accuracy : {np.mean(fold_accuracy):.4f}")
print(f"Average Precision: {np.mean(fold_precision):.4f}")
print(f"Average Recall   : {np.mean(fold_recall):.4f}")
print(f"Average Macro F1 : {np.mean(fold_f1):.4f}")

print()

print(f"Std Accuracy : {np.std(fold_accuracy):.4f}")
print(f"Std Macro F1 : {np.std(fold_f1):.4f}")

5-Fold Cross Validation Results
Average Accuracy : 0.8121
Average Precision: 0.8026
Average Recall   : 0.8187
Average Macro F1 : 0.8082

Std Accuracy : 0.0058
Std Macro F1 : 0.0056


In [ ]:
# Find the Best Fold
best_fold = np.argmax(fold_f1)

print(f"\nBest Fold : {best_fold+1}")
print(f"Best Macro F1 : {fold_f1[best_fold]:.4f}")


Best Fold : 4
Best Macro F1 : 0.8144


In [ ]:
# Copy the Best Model
import shutil

best_model = os.path.join(
    SAVE_DIR,
    f"Fold_{best_fold+1}"
)

final_model = os.path.join(
    SAVE_DIR,
    "Best_Model"
)

if os.path.exists(final_model):
    shutil.rmtree(final_model)

shutil.copytree(best_model, final_model)

print("Best model copied successfully!")

Best model copied successfully!


In [ ]:
# Load Best Model
BEST_MODEL = os.path.join(SAVE_DIR, "Best_Model")

tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL)

model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL)

model.to(device)

model.eval()

print("Best model loaded successfully.")

Best model loaded successfully.


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ================================================================
# Reload the best cross-validation model, for standalone inference
# ================================================================

MODEL_PATH = "/content/drive/MyDrive/XLM-R Cross Validation/XLMR_CV/Best_Model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

print("Model Loaded Successfully!")

# ==========================================
# Label Mapping
# ==========================================

label_map = {
    0: "AGAINST",
    1: "FAVOR",
    2: "NONE"
}

# ==========================================
# Prediction Function
# ==========================================

def predict(text):

    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():

        outputs = model(**inputs)

        pred = torch.argmax(outputs.logits, dim=1).item()

    return label_map[pred]

Model Loaded Successfully!


In [ ]:
# Run predictions on the Hindi test set and save the results
hindi_test = pd.read_excel("/content/Hindi_500_test data.xlsx")

print(hindi_test.head())

hindi_test["Predicted_Label"] = hindi_test["COMMENT"].apply(predict)

hindi_test.to_excel(
    "/content/Hindi_500_Predictions_CV.xlsx",
    index=False
)

print("Hindi Prediction Saved Successfully!")

   ID                                            COMMENT
0   1  बिल नाई एक वैज्ञानिक नहीं हैं। वह एक यांत्रिक ...
1   2           ऑनलाइन कक्षाओं के लिए और कौन उपस्थित है?
2   3  शाकाहारियों को इतना परेशान करने वाला क्यों होत...
3   4                      विवरण में बिल न्ये मर्च लिंक।
4   5  बिल के प्रति इतना घृणा क्यों है? मुझे समझ नहीं...
Hindi Prediction Saved Successfully!


In [ ]:
# Run predictions on the Bangla test set and save the results
bangla_test = pd.read_excel("/content/Bangla_500_test data.xlsx")

print(bangla_test.head())

bangla_test["Predicted_Label"] = bangla_test["COMMENT"].apply(predict)

bangla_test.to_excel(
    "/content/Bangla_500_Predictions_CV.xlsx",
    index=False
)

print("Bangla Prediction Saved Successfully!")

   ID                                            COMMENT
0   1  পৃথিবীর নৃত্য এবং অনন্ত আকাশের মধ্যে, ফিসফিস ক...
1   2  ধন্যবাদ, আপনার করা সমস্ত কিছুর জন্য এবং আপনি য...
2   3  এটি বায়ুমণ্ডলে কার্বন ডাই-অক্সাইডের ২৫০ ডিগ্র...
3   4  জলবায়ু-পরিবর্তক দেবতা-পাথর জানে যে পৃথিবীর দু...
4   5  আমার শিক্ষক আমাকে বলেছিলেন যে প্রতি ১০০ বছরে প...
Bangla Prediction Saved Successfully!
